# App1: Reaction Load Params - REST API to JSON Schema

**Workspace ID:** 2672  
**Entity ID:** 12162  
**App URL:** https://demo.viktor.ai/workspaces/2672/app/editor/12162

This notebook reads the entity data through the VIKTOR REST API and converts the saved parameters/properties to JSON Schema.

In [ ]:
import os
import json
from pathlib import Path
from typing import Any

import requests
from dotenv import find_dotenv, load_dotenv

DOTENV_PATH = find_dotenv(usecwd=True)
load_dotenv(DOTENV_PATH, override=True)

print(f"Loaded .env: {DOTENV_PATH or '<not found>'}")


def get_env_int(*names: str, default: int) -> int:
    for name in names:
        raw_value = os.getenv(name)
        if raw_value and raw_value.strip():
            try:
                return int(raw_value)
            except ValueError as exc:
                raise ValueError(f"{name} must be an integer.") from exc
    return default



WORKSPACE_ID = get_env_int("APP1_WORKSPACE_ID", "VIKTOR_WORKSPACE_ID", default=2672)
ENTITY_ID = get_env_int("APP1_ENTITY_ID", "VIKTOR_ENTITY_ID", default=12162)
APP_OUTPUT_DIR = Path("app1")
OUTPUT_FILE = APP_OUTPUT_DIR / "input_schema.json"
METHODS_OUTPUT_FILE = APP_OUTPUT_DIR / "available_methods.json"
METHOD_RESULT_OUTPUT_FILE = APP_OUTPUT_DIR / "first_method_result.json"

print(f"Target: Workspace {WORKSPACE_ID}, Entity {ENTITY_ID}")

## 1. Connect to VIKTOR REST API

The REST helper below follows the repository VIKTOR REST API rules: normalize the API base once, read the token from environment variables, do not print the token, centralize timeouts and errors, and pass query parameters through `params=...`.

In [ ]:
# Initialize a small REST API client with a Personal Access Token.
# Supported token environment variables, in order:
# TOKEN_VK_APP, VIKTOR_TOKEN, VIKTOR_API_TOKEN.

TOKEN_ENV_NAMES = ("TOKEN_VK_APP", "VIKTOR_TOKEN", "VIKTOR_API_TOKEN")


def normalize_bearer_token(raw_token: str, *, env_var: str) -> str:
    token = raw_token.strip().strip('"').strip("'").strip()

    if token.startswith(f"{env_var}="):
        token = token.split("=", 1)[1].strip().strip('"').strip("'").strip()

    if token.lower().startswith("authorization:"):
        token = token.split(":", 1)[1].strip()

    if token.lower().startswith("bearer "):
        token = token.split(None, 1)[1].strip()

    token = token.strip().strip('"').strip("'").strip()
    if not token:
        raise ValueError(f"{env_var} is empty.")
    if any(ch.isspace() for ch in token):
        raise ValueError(f"{env_var} contains whitespace. Paste only the token value.")
    return token


def get_token() -> tuple[str, str]:
    for env_var in TOKEN_ENV_NAMES:
        raw_token = os.getenv(env_var)
        if raw_token and raw_token.strip():
            return normalize_bearer_token(raw_token, env_var=env_var), env_var
    raise ValueError(
        "Missing VIKTOR token. Set TOKEN_VK_APP or VIKTOR_TOKEN. "
        "VIKTOR_API_TOKEN is also accepted for backwards compatibility."
    )


def get_api_base() -> str:
    api_base = os.getenv("VIKTOR_API_BASE")
    if api_base and api_base.strip():
        base = api_base.strip().rstrip("/")
    else:
        environment = os.getenv("VIKTOR_ENVIRONMENT", "demo").strip().rstrip("/")
        if not environment:
            raise ValueError("Missing VIKTOR_ENVIRONMENT.")
        if environment.startswith("https://"):
            base = environment
        elif environment.startswith("http://"):
            base = "https://" + environment.removeprefix("http://")
        elif environment.endswith(".viktor.ai"):
            base = f"https://{environment}"
        else:
            base = f"https://{environment}.viktor.ai"

    base = base.rstrip("/")
    if base.startswith("http://"):
        base = "https://" + base.removeprefix("http://")
    if not base.startswith("https://"):
        raise ValueError("VIKTOR API base must resolve to an HTTPS URL.")
    return base if base.endswith("/api") else f"{base}/api"


class ViktorRestClient:
    def __init__(
        self,
        *,
        api_base: str,
        token: str,
        connect_timeout: float = 5.0,
        read_timeout: float = 30.0,
    ) -> None:
        token = token.strip()
        if not token:
            raise ValueError("Missing VIKTOR token.")

        base = api_base.strip().rstrip("/")
        self.api_base = base if base.endswith("/api") else f"{base}/api"
        self.timeout = (connect_timeout, read_timeout)
        self.session = requests.Session()
        self.session.headers.update(
            {
                "Authorization": f"Bearer {token}",
                "Accept": "application/json",
            }
        )

    def url(self, path_or_url: str) -> str:
        if path_or_url.startswith("http://") or path_or_url.startswith("https://"):
            return path_or_url
        return f"{self.api_base}/{path_or_url.lstrip('/')}"

    def check_response(self, response: requests.Response, *, action: str) -> None:
        if response.ok:
            return
        body = response.text[:500]
        raise RuntimeError(f"{action} failed (status={response.status_code}): {body}")

    def request_json(
        self,
        method: str,
        path_or_url: str,
        *,
        params: dict[str, Any] | None = None,
        json_body: dict[str, Any] | None = None,
        action: str = "VIKTOR REST request",
    ) -> dict[str, Any]:
        response = self.session.request(
            method=method.upper(),
            url=self.url(path_or_url),
            params=params,
            json=json_body,
            timeout=self.timeout,
        )
        self.check_response(response, action=action)
        if not response.content:
            return {}
        try:
            return response.json()
        except ValueError as exc:
            raise RuntimeError(
                f"{action} did not return valid JSON: {response.text[:500]}"
            ) from exc

    def get_json(
        self,
        path_or_url: str,
        *,
        params: dict[str, Any] | None = None,
        action: str = "GET request",
    ) -> dict[str, Any]:
        return self.request_json("GET", path_or_url, params=params, action=action)


    def post_json(
        self,
        path_or_url: str,
        *,
        json_body: dict[str, Any] | None = None,
        action: str = "POST request",
    ) -> dict[str, Any]:
        return self.request_json("POST", path_or_url, json_body=json_body, action=action)


    def create_job(
        self,
        *,
        workspace_id: int,
        entity_id: int,
        method_name: str,
        params: dict[str, Any],
    ) -> dict[str, Any]:
        job = self.post_json(
            f"workspaces/{workspace_id}/entities/{entity_id}/jobs/",
            json_body={
                "method_name": method_name,
                "params": params,
                "poll_result": False,
            },
            action="Create VIKTOR job",
        )
        if job.get("url"):
            return self.poll_job(job["url"])
        if job.get("status") == "success":
            return job
        raise RuntimeError(f"Unexpected VIKTOR job response: {job}")

    def poll_job(self, job_url: str) -> dict[str, Any]:
        import time

        failed_statuses = {"failed", "cancelled", "error", "error_user", "error_timeout"}
        deadline = time.monotonic() + 300
        sleep_seconds = 0.8

        while time.monotonic() < deadline:
            job = self.get_json(job_url, action="Poll VIKTOR job")
            status = job.get("status")
            if status == "success":
                return job
            if status in failed_statuses:
                raise RuntimeError(f"VIKTOR job failed with status={status}: {job.get('error')}")
            time.sleep(sleep_seconds)
            sleep_seconds = min(sleep_seconds * 1.5, 5.0)

        raise TimeoutError("VIKTOR job did not finish within 300 seconds.")

    def compute_method(
        self,
        *,
        workspace_id: int,
        entity_id: int,
        method_name: str,
        params: dict[str, Any],
    ) -> dict[str, Any]:
        job = self.create_job(
            workspace_id=workspace_id,
            entity_id=entity_id,
            method_name=method_name,
            params=params,
        )
        result = job.get("result") or job.get("content")
        if not isinstance(result, dict):
            raise RuntimeError(f"VIKTOR job did not return a JSON object result: {job}")
        return result

    def build_entity_editor_url(self, *, workspace_id: int, entity_id: int) -> str:
        ui_base = self.api_base[:-4] if self.api_base.endswith("/api") else self.api_base
        return f"{ui_base}/workspaces/{workspace_id}/app/editor/{entity_id}"


api_token, token_env_var = get_token()
client = ViktorRestClient(api_base=get_api_base(), token=api_token)

print(f"Using VIKTOR token from environment variable: {token_env_var}")
print(f"REST API base URL: {client.api_base}")

workspaces = client.get_json(
    "workspaces/",
    params={"limit": 1, "offset": 0, "detail_level": "minimal"},
    action="List workspaces",
)
if "results" not in workspaces:
    raise RuntimeError(f"Unexpected workspace list response keys: {list(workspaces.keys())}")

print("Connected to VIKTOR REST API")
print(f"Workspace list response keys: {list(workspaces.keys())}")

## 2. Fetch Entity and Parametrization

In [ ]:
# Fetch the entity through REST.
entity = client.get_json(
    f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/",
    params={
        "properties": "true",
        "clean_params": "true",
        "param_types": "true",
    },
    action="Get entity",
)

print(f"Entity: {entity.get('name', '<no name returned>')}")
print(f"Type: {entity.get('entity_type_name', entity.get('entity_type', '<no type returned>'))}")
print(f"Editor URL: {client.build_entity_editor_url(workspace_id=WORKSPACE_ID, entity_id=ENTITY_ID)}")
print(f"Response keys: {list(entity.keys())}")


def extract_saved_params(entity_payload: dict[str, Any]) -> dict[str, Any]:
    """Extract saved params/properties from a VIKTOR REST entity response."""
    for key in ("properties", "params", "last_saved_params"):
        value = entity_payload.get(key)
        if isinstance(value, dict):
            return value

    for key in ("last_saved_revision", "latest_revision", "revision"):
        value = entity_payload.get(key)
        if isinstance(value, dict) and isinstance(value.get("params"), dict):
            return value["params"]

    raise KeyError(
        "No saved params/properties found in the REST response. "
        f"Available keys: {list(entity_payload.keys())}"
    )


params = extract_saved_params(entity)
print(f"\nParameter keys: {list(params.keys()) if params else 'None'}")

## 3. Get Available Methods

The entity type exposes view controller methods. The editor parametrization payload can also expose button/action methods.

In [ ]:
def set_nested_default(target: dict[str, Any], dotted_path: str, value: Any) -> None:
    keys = [part for part in dotted_path.split(".") if part]
    if not keys:
        return
    current = target
    for key in keys[:-1]:
        child = current.setdefault(key, {})
        if not isinstance(child, dict):
            child = {}
            current[key] = child
        current = child
    current[keys[-1]] = value


def collect_declared_defaults(nodes: Any) -> dict[str, Any]:
    defaults: dict[str, Any] = {}

    def walk(value: Any) -> None:
        if isinstance(value, list):
            for item in value:
                walk(item)
            return
        if not isinstance(value, dict):
            return

        path = value.get("name") or value.get("parametrization_path")
        if path and "default" in value:
            set_nested_default(defaults, path, value["default"])

        for child_key in ("content", "children", "items"):
            walk(value.get(child_key))

    walk(nodes)
    return defaults


def deep_merge(defaults: Any, overrides: Any) -> Any:
    if isinstance(defaults, dict) and isinstance(overrides, dict):
        merged = dict(defaults)
        for key, value in overrides.items():
            merged[key] = deep_merge(merged.get(key), value)
        return merged
    if overrides is not None:
        return overrides
    return defaults


def collect_view_methods(entity_type_payload: dict[str, Any]) -> list[dict[str, Any]]:
    methods: list[dict[str, Any]] = []
    for view in entity_type_payload.get("views") or []:
        if not isinstance(view, dict):
            continue
        method = view.get("controller_method") or view.get("method_name")
        if method:
            methods.append(
                {
                    "method_name": method,
                    "source": "entity_type.views",
                    "label": view.get("label"),
                    "view_type": view.get("view_type"),
                    "automatic_update": view.get("automatic_update"),
                }
            )
    return methods


def collect_parametrization_methods(nodes: Any) -> list[dict[str, Any]]:
    methods: list[dict[str, Any]] = []

    def walk(value: Any) -> None:
        if isinstance(value, list):
            for item in value:
                walk(item)
            return
        if not isinstance(value, dict):
            return

        method = value.get("method")
        if method:
            methods.append(
                {
                    "method_name": method,
                    "source": "parametrization",
                    "label": value.get("ui_name") or value.get("title"),
                    "node_type": value.get("type"),
                    "path": value.get("name") or value.get("parametrization_path"),
                }
            )

        for child_key in ("content", "children", "items"):
            walk(value.get(child_key))

    walk(nodes)
    return methods


def deduplicate_methods(methods: list[dict[str, Any]]) -> list[dict[str, Any]]:
    deduped: dict[str, dict[str, Any]] = {}
    for item in methods:
        deduped[item["method_name"]] = item
    return list(deduped.values())


entity_type = client.get_json(
    f"workspaces/{WORKSPACE_ID}/entity_types/{entity['entity_type']}/",
    action="Get entity type",
)

editor_session = client.post_json(
    f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/session/",
    action="Create editor session",
)

parametrization = client.post_json(
    f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/parametrization/",
    json_body={
        "editor_session": editor_session["editor_session"],
        "params": {},
    },
    action="Get parametrization",
)

parametrization_nodes = (parametrization.get("content") or {}).get("parametrization", [])
declared_defaults = collect_declared_defaults(parametrization_nodes)
effective_params = deep_merge(declared_defaults, params)
params = effective_params

available_methods = deduplicate_methods(
    collect_view_methods(entity_type) + collect_parametrization_methods(parametrization_nodes)
)

methods_output_path = Path(METHODS_OUTPUT_FILE)
methods_output_path.parent.mkdir(parents=True, exist_ok=True)
with open(methods_output_path, "w", encoding="utf-8") as f:
    json.dump(available_methods, f, indent=2)

print(f"Declared default top-level keys: {list(declared_defaults.keys())}")
print(f"Effective parameter keys: {list(params.keys())}")
print(f"Available methods: {len(available_methods)}")
print(json.dumps(available_methods, indent=2, default=str))
print(f"Methods saved to: {methods_output_path.absolute()}")

## 4. Execute First Available View Method

The notebook calls the first available DataView/TableView-style method with the effective params and saves the selected result payload. It uses `POST /api/workspaces/{workspace_id}/entities/{entity_id}/jobs/` with `method_name`, `params`, and `poll_result: false`, then polls the returned `url` until `status == "success"`.

Result payload keys can include `data`, `table`, `download`, `geometry`, `plotly`, `geojson`, `web`, `pdf`, `image`, `ifc`, `optimization`, and `set_params`.

In [ ]:
RESULT_KEY_PRIORITY = [
    "data",
    "table",
    "download",
    "geometry",
    "plotly",
    "geojson",
    "web",
    "pdf",
    "image",
    "ifc",
    "optimization",
    "set_params",
]

# VIKTOR method result payload keys can include:
# data, table, download, geometry, plotly, geojson, web, pdf, image, ifc, optimization, set_params.


def pick_first_data_or_view_method(methods: list[dict[str, Any]]) -> dict[str, Any]:
    for method in methods:
        if method.get("view_type") in {"data", "table"}:
            return method
    for method in methods:
        if method.get("source") == "entity_type.views" or method.get("view_type"):
            return method
    if methods:
        return methods[0]
    raise RuntimeError("No available VIKTOR methods found to execute.")


def select_first_result_payload(result: dict[str, Any]) -> tuple[str, Any]:
    for key in RESULT_KEY_PRIORITY:
        if key in result:
            return key, result[key]
    return "result", result


selected_method = pick_first_data_or_view_method(available_methods)
method_result = client.compute_method(
    workspace_id=WORKSPACE_ID,
    entity_id=ENTITY_ID,
    method_name=selected_method["method_name"],
    params=params,
)
result_key, selected_result_payload = select_first_result_payload(method_result)

method_result_output = {
    "method": selected_method,
    "result_key": result_key,
    "result": selected_result_payload,
}

method_result_output_path = Path(METHOD_RESULT_OUTPUT_FILE)
method_result_output_path.parent.mkdir(parents=True, exist_ok=True)
with open(method_result_output_path, "w", encoding="utf-8") as f:
    json.dump(method_result_output, f, indent=2)

print(f"Executed method: {selected_method['method_name']}")
print(f"Selected result key: {result_key}")
print(f"Method result saved to: {method_result_output_path.absolute()}")

## 5. Infer JSON Schema From Effective Params

The schema is inferred from declared defaults merged with saved entity properties. Saved properties win over declared defaults.

In [ ]:
def infer_json_schema_from_params(
    params: dict[str, Any],
    *,
    include_defaults: bool = True,
) -> dict[str, Any]:
    schema: dict[str, Any] = {
        "type": "object",
        "properties": {},
        "additionalProperties": False,
    }
    required: list[str] = []

    for key, value in params.items():
        property_schema = infer_type_from_value(value, include_defaults=include_defaults)
        if include_defaults:
            property_schema["default"] = value
        else:
            required.append(key)
        schema["properties"][key] = property_schema

    if required:
        schema["required"] = required
    return schema


def infer_type_from_value(value: Any, *, include_defaults: bool = True) -> dict[str, Any]:
    if isinstance(value, bool):
        return {"type": "boolean"}
    if isinstance(value, int) and not isinstance(value, bool):
        return {"type": "integer"}
    if isinstance(value, float):
        return {"type": "number"}
    if isinstance(value, str):
        return {"type": "string"}
    if isinstance(value, list):
        return {
            "type": "array",
            "items": infer_array_item_schema(value),
        }
    if isinstance(value, dict):
        return infer_json_schema_from_params(value, include_defaults=include_defaults)
    if value is None:
        return {"type": ["string", "null"]}
    return {"type": "string"}


def infer_array_item_schema(values: list[Any]) -> dict[str, Any]:
    if not values:
        return {"type": "string"}
    if all(isinstance(item, dict) for item in values):
        merged_shape: dict[str, Any] = {}
        for item in values:
            merged_shape = deep_merge(merged_shape, item)
        return infer_json_schema_from_params(merged_shape, include_defaults=False)
    return infer_type_from_value(values[0], include_defaults=False)


if params:
    schema = infer_json_schema_from_params(params)
    print("Schema generated with defaults")
    print(f"\nProperties: {list(schema['properties'].keys())}")
else:
    print("No parameters found; creating an empty schema")
    schema = {
        "type": "object",
        "properties": {},
        "additionalProperties": False,
    }

## 6. Preview the Schema

In [ ]:
print(json.dumps(schema, indent=2))

## 7. Save Schema to JSON File

In [ ]:
output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

print(f"Schema saved to: {output_path.absolute()}")

## 8. Display Effective Parameters

These are the declared defaults after applying saved entity properties on top.

In [ ]:
print("Effective parameters used for schema defaults:")
print(json.dumps(params, indent=2, default=str))